# What Does Your Diversifier Do? — companion notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kris-SF/data-pipelines/blob/main/quant-analysis/diversifier_study.ipynb) &nbsp;&nbsp;&nbsp; [GitHub repo](https://github.com/Kris-SF/data-pipelines/tree/main/quant-analysis)

Every empirical number in the post comes out of this notebook. It downloads the data when you run it (nothing is committed), builds one monthly panel of ETF returns, cash and inflation breakevens, then produces the post's tables in order: the ETF Sharpe-hurdle table, the weak-bond-year screen, conditional stats in rising and falling breakeven months, the stability cuts, and a block bootstrap of the conditional Sharpe difference.

Two modes, set in the config cell below:

- **`reproduce`** — the fixed Aug 2006 – Jul 2026 window used in the post (240 monthly returns). After running, a side-by-side table compares every computed number against the published value with a pass/flag column.
- **`refresh`** — same start date and methodology, but the window ends at the latest month that is complete in every series. The notebook prints which series is the binding constraint (usually the French RF file).

All logic lives in `diversifier.py`; this notebook only calls into it. Set `MODE`, then **Run All**. Works in Google Colab (one-click) and locally in VS Code / Jupyter.

In [ ]:
# Bootstrap — runs in both Colab and local Jupyter/VS Code.
# In Colab: clones the repo (so hurst.py / data.py are importable) and installs deps.
# Local: no-op. Run the commented pip line once if you haven't installed requirements yet.
try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os
    if not os.path.isdir("data-pipelines"):
        !git clone --quiet --depth 1 https://github.com/Kris-SF/data-pipelines.git
    %cd data-pipelines/quant-analysis
    %pip install --quiet -r requirements.txt

# Local first-time install (uncomment once, then comment back):
# %pip install -r requirements.txt

In [ ]:
MODE = "reproduce"   # "reproduce" = fixed Aug 2006 – Jul 2026 window, diffed against the post
                     # "refresh"   = same start, ends at the latest month complete in every series

In [ ]:
from diversifier import StudyConfig, run_study, compare_to_reported, display_comparison

cfg = StudyConfig.for_mode(MODE)
result = run_study(cfg)   # downloads, builds the panel, computes and displays every table, writes output/<MODE>/

In [ ]:
# Published values from the post. These are the test, not the target.
# Tolerances: ±0.005 Sharpes/correlations, ±0.02 pp monthly means/vols, ±0.01 bootstrap endpoints,
# exact match on counts and year sets.
REPORTED = {
    # ticker: (annualized Sharpe, correlation with SPY, 80/20 annualized Sharpe)
    "etf_table": {
        "SPY": (0.675, 1.000, 0.675),
        "GLD": (0.511, 0.076, 0.773),
        "IEF": (0.288, -0.067, 0.707),
        "TLT": (0.167, -0.080, 0.707),
        "AGG": (0.349, 0.245, 0.686),
        "DBC": (0.109, 0.440, 0.606),
        "USO": (-0.028, 0.326, 0.491),
    },
    "dbc_hurdle": 0.297,                        # rho × S_SPY; DBC and USO both fail
    "weak_bond_years": [2013, 2015, 2018, 2021, 2022],
    "weak_bond_wins": {"DBC": 2, "USO": 3},     # years 60/30/10 beat 60/40, out of 5
    # values are (60/40, 60/30/10 DBC, 60/30/10 USO)
    "rising_10y": {
        "n": 122,
        "mean_excess_pct": (1.23, 1.55, 1.70),
        "vol_pct": (2.64, 2.66, 2.78),
        "sharpe": (0.468, 0.581, 0.610),
    },
    "falling_10y": {
        "n": 114,
        "n_unchanged": 4,
        "mean_excess_pct": (-0.07, -0.39, -0.60),
        "vol_pct": (2.88, 3.11, 3.34),
        "sharpe": (-0.024, -0.126, -0.179),
    },
    "full_sample_sharpe": (0.201, 0.189, 0.168),   # 240 months, not annualized
    # cut: (rising-month count, 60/40, 60/30/10 DBC, 60/30/10 USO)
    "stability": {
        "10y_full":        (122, 0.468, 0.581, 0.610),
        "10y_first_half":  (58,  0.595, 0.678, 0.643),   # Aug 2006 – Jul 2016
        "10y_second_half": (64,  0.366, 0.499, 0.578),   # Aug 2016 – Jul 2026
        "5y_full":         (122, 0.419, 0.506, 0.527),
    },
    "bootstrap_point": {"DBC": 0.114, "USO": 0.143},
    # config: {etf: (2.5th pct, 97.5th pct)} of the conditional Sharpe difference
    "bootstrap_ci": {
        "10y_L3":  {"DBC": (0.075, 0.154), "USO": (0.072, 0.228)},
        "10y_L6":  {"DBC": (0.074, 0.153), "USO": (0.071, 0.227)},
        "10y_L12": {"DBC": (0.073, 0.152), "USO": (0.070, 0.227)},
        "5y_L6":   {"DBC": (0.045, 0.139), "USO": (0.031, 0.204)},
    },
}

if MODE == "reproduce":
    diff = compare_to_reported(result, REPORTED)
    display_comparison(diff)
    diff.to_csv(result.run_dir / "reproduction_check.csv", index=False)
else:
    print("refresh mode: no comparison against the published values (the window differs).")

`result` holds everything for further poking:
- `result.panel.frame()` — the 240-row monthly panel (ETF returns, RF, breakeven changes in bp)
- `result.panel.price_dates` — the actual price date used for each ticker and month
- `result.etf`, `result.weak_bond`, `result.cond_10y`, `result.full_sharpes`, `result.stability`, `result.bootstrap` — each table as a DataFrame / dict
- `result.tables` — the same numbers as written to `output/<MODE>/post_tables.json`

Output folder (`output/<MODE>/`, git-ignored): `post_tables.json`, `price_dates.csv`, `monthly_panel.csv`, and `reproduction_check.csv` in reproduce mode. Downloaded data is cached for the day in `data_cache/` (also git-ignored).